# Manual QC Inspector

Interactive notebook for visually inspecting PIPS data and marking time ranges that need manual quality control.

**Workflow:**
1. Run cells to set up environment and load configuration
2. Use `next_dataset()` to load each PIPS/IOP combination
3. Examine plots - zoom in on suspicious regions using interactive controls
4. Note exact start/end times of problematic periods
5. Use `add_manual_qc()` to record QC decisions
6. Use `save_manual_qc()` periodically to save your work
7. Continue with `next_dataset()` until all datasets are inspected
8. Apply decisions using `apply_manual_qc.py` script

In [ ]:
# Setup
%matplotlib widget
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import json
from datetime import datetime
from pathlib import Path
import sys

# Add parent directory to path for importing configs
sys.path.insert(0, '..')

In [ ]:
# Configuration - EDIT THIS SECTION
# Specify your case configuration file
CONFIG_FILE = '../configs/ICECHIP_IOP9_2025_10s.py'  # Example: adjust to your needs

# Output file for manual QC decisions
MANUAL_QC_FILE = '../configs/manual_qc_decisions.json'

# Load case configuration using pyPIPS utils
import pyPIPS.utils as utils

config = utils.import_all_from(CONFIG_FILE)
PIPS_IO_dict = config.PIPS_IO_dict

PIPS_dir = PIPS_IO_dict['PIPS_dir']
deployment_names = PIPS_IO_dict['deployment_names']
PIPS_names = PIPS_IO_dict['PIPS_names']

print(f"Configuration loaded: {CONFIG_FILE}")
print(f"PIPS directory: {PIPS_dir}")
print(f"Deployments: {len(deployment_names)}")
print(f"PIPS stations: {PIPS_names}")

In [ ]:
# Load or initialize manual QC tracking dictionary
try:
    with open(MANUAL_QC_FILE, 'r') as f:
        manual_qc_dict = json.load(f)
    print(f"Loaded existing manual QC decisions from {MANUAL_QC_FILE}")
    print(f"Total entries: {sum(len(v) for v in manual_qc_dict.values())}")
except FileNotFoundError:
    manual_qc_dict = {}
    print("Starting new manual QC dictionary")

# Initialize manual trimming dictionary (separate from variable QC)
try:
    # Try to load from same file if it has a 'manual_trim' key
    with open(MANUAL_QC_FILE, 'r') as f:
        data = json.load(f)
        manual_trim_dict = data.get('_manual_trim', {})
    if manual_trim_dict:
        print(f"Loaded {len(manual_trim_dict)} manual trim entries")
except (FileNotFoundError, json.JSONDecodeError):
    manual_trim_dict = {}
    print("Starting new manual trim dictionary")

In [ ]:
# Helper functions for managing QC decisions

def add_manual_qc(pips_name, deployment_name, variable, start_time, end_time, reason=""):
    """
    Add a manual QC decision to the tracking dictionary.

    Parameters
    ----------
    pips_name : str
        PIPS station name (e.g., 'PIPS1A')
    deployment_name : str
        Deployment/IOP name (e.g., 'IOP1_2025')
    variable : str
        Variable to QC (e.g., 'slowtemp')
    start_time : str or pd.Timestamp
        Start time of QC period (ISO format)
    end_time : str or pd.Timestamp
        End time of QC period (ISO format)
    reason : str, optional
        Reason for QC (e.g., 'sensor malfunction', 'anomalous spike')
    """
    key = f"{pips_name}_{deployment_name}"

    if key not in manual_qc_dict:
        manual_qc_dict[key] = []

    # Convert timestamps to ISO format strings
    if isinstance(start_time, pd.Timestamp):
        start_time = start_time.isoformat()
    if isinstance(end_time, pd.Timestamp):
        end_time = end_time.isoformat()

    qc_entry = {
        'variable': variable,
        'start_time': start_time,
        'end_time': end_time,
        'reason': reason,
        'flagged_on': datetime.now().isoformat()
    }

    manual_qc_dict[key].append(qc_entry)
    print(f"\n✓ Added QC: {pips_name}/{deployment_name} - {variable}")
    print(f"  Time range: {start_time} to {end_time}")
    print(f"  Reason: {reason}")
    print(f"  Total entries for this dataset: {len(manual_qc_dict[key])}")


def save_manual_qc():
    """Save manual QC and trim decisions to JSON file."""
    # Combine both dictionaries into one file
    # Use special key '_manual_trim' to avoid conflicts with PIPS_IOP keys
    output_data = manual_qc_dict.copy()
    if manual_trim_dict:
        output_data['_manual_trim'] = manual_trim_dict

    with open(MANUAL_QC_FILE, 'w') as f:
        json.dump(output_data, f, indent=2)

    total_qc_entries = sum(len(v) for v in manual_qc_dict.values())
    total_trim_entries = len(manual_trim_dict)
    print(f"\n✓ Saved to {MANUAL_QC_FILE}")
    print(f"  {total_qc_entries} manual QC entries")
    print(f"  {total_trim_entries} manual trim entries")


def view_manual_qc(pips_name=None, deployment_name=None):
    """View existing manual QC decisions."""
    if pips_name and deployment_name:
        key = f"{pips_name}_{deployment_name}"
        if key in manual_qc_dict and len(manual_qc_dict[key]) > 0:
            print(f"\nManual QC for {key}:")
            for i, entry in enumerate(manual_qc_dict[key], 1):
                print(f"  {i}. {entry['variable']}: {entry['start_time']} to {entry['end_time']}")
                print(f"     Reason: {entry['reason']}")
        else:
            print(f"No manual QC entries for {key}")
    else:
        if len(manual_qc_dict) == 0:
            print("No manual QC decisions recorded yet")
        else:
            print("\nAll manual QC decisions:")
            for key, entries in manual_qc_dict.items():
                print(f"\n{key}: {len(entries)} entries")
                for i, entry in enumerate(entries, 1):
                    print(f"  {i}. {entry['variable']}: {entry['start_time']} to {entry['end_time']}")
                    if entry['reason']:
                        print(f"     Reason: {entry['reason']}")


def delete_last_qc(pips_name, deployment_name):
    """Remove the most recently added QC entry for a dataset."""
    key = f"{pips_name}_{deployment_name}"
    if key in manual_qc_dict and len(manual_qc_dict[key]) > 0:
        removed = manual_qc_dict[key].pop()
        print(f"Removed QC entry: {removed['variable']} {removed['start_time']} to {removed['end_time']}")
    else:
        print(f"No QC entries to remove for {key}")


def add_manual_trim(pips_name, deployment_name, new_start_time=None, new_end_time=None, reason=""):
    """
    Add manual trimming decision to trim entire dataset to new time range.
    Updates dataset start and/or end times, similar to automated trimming in apply_QC.py.

    Parameters
    ----------
    pips_name : str
        PIPS station name (e.g., 'PIPS1A')
    deployment_name : str
        Deployment/IOP name (e.g., 'IOP1_2025')
    new_start_time : str or pd.Timestamp or None
        New start time for dataset (None = keep original start)
    new_end_time : str or pd.Timestamp or None
        New end time for dataset (None = keep original end)
    reason : str, optional
        Reason for trimming (e.g., 'Remove deployment period', 'Bad data at start')
    """
    if new_start_time is None and new_end_time is None:
        print("Error: Must specify at least one of new_start_time or new_end_time")
        return

    key = f"{pips_name}_{deployment_name}"

    # Convert timestamps to ISO format strings
    if new_start_time is not None and isinstance(new_start_time, pd.Timestamp):
        new_start_time = new_start_time.isoformat()
    if new_end_time is not None and isinstance(new_end_time, pd.Timestamp):
        new_end_time = new_end_time.isoformat()

    trim_entry = {
        'new_start_time': new_start_time,
        'new_end_time': new_end_time,
        'reason': reason,
        'trimmed_on': datetime.now().isoformat()
    }

    manual_trim_dict[key] = trim_entry  # Only one trim decision per dataset

    print(f"\n✓ Added trimming for {pips_name}/{deployment_name}")
    if new_start_time:
        print(f"  New start time: {new_start_time}")
    if new_end_time:
        print(f"  New end time: {new_end_time}")
    if reason:
        print(f"  Reason: {reason}")


def view_manual_trim(pips_name=None, deployment_name=None):
    """View existing manual trim decisions."""
    if pips_name and deployment_name:
        key = f"{pips_name}_{deployment_name}"
        if key in manual_trim_dict:
            entry = manual_trim_dict[key]
            print(f"\nManual trim for {key}:")
            if entry.get('new_start_time'):
                print(f"  New start: {entry['new_start_time']}")
            if entry.get('new_end_time'):
                print(f"  New end: {entry['new_end_time']}")
            if entry.get('reason'):
                print(f"  Reason: {entry['reason']}")
        else:
            print(f"No manual trim for {key}")
    else:
        if len(manual_trim_dict) == 0:
            print("No manual trim decisions recorded yet")
        else:
            print("\nAll manual trim decisions:")
            for key, entry in manual_trim_dict.items():
                print(f"\n{key}:")
                if entry.get('new_start_time'):
                    print(f"  New start: {entry['new_start_time']}")
                if entry.get('new_end_time'):
                    print(f"  New end: {entry['new_end_time']}")
                if entry.get('reason'):
                    print(f"  Reason: {entry['reason']}")


def delete_manual_trim(pips_name, deployment_name):
    """Remove manual trim decision for a dataset."""
    key = f"{pips_name}_{deployment_name}"
    if key in manual_trim_dict:
        removed = manual_trim_dict.pop(key)
        print(f"Removed trim for {key}")
        if removed.get('new_start_time'):
            print(f"  Start: {removed['new_start_time']}")
        if removed.get('new_end_time'):
            print(f"  End: {removed['new_end_time']}")
    else:
        print(f"No trim entry to remove for {key}")


print("Helper functions loaded:")
print("  add_manual_qc(pips, iop, variable, start, end, reason)")
print("  save_manual_qc()")
print("  view_manual_qc([pips], [iop])")
print("  delete_last_qc(pips, iop)")
print("  ")
print("  add_manual_trim(pips, iop, new_start, new_end, reason)")
print("  view_manual_trim([pips], [iop])")
print("  delete_manual_trim(pips, iop)")

In [ ]:
# Interactive inspection function with click-to-select time ranges

# Global variables to store selected times
selected_start_time = None
selected_end_time = None

def inspect_variable(pips_name, deployment_name, variable='slowtemp',
                     compare_with='fasttemp', window=None, figsize=(11, 7)):
    """
    Interactively inspect a variable and mark time ranges for QC.

    **Interactive Controls:**
    - **Hover**: See exact timestamp under cursor
    - **Left Click**: Set start time (first click) or end time (second click)
    - **Right Click**: Clear selected times

    After selecting times, use the printed variables in add_manual_qc()

    Parameters
    ----------
    pips_name : str
        PIPS station name
    deployment_name : str
        Deployment/IOP name
    variable : str
        Primary variable to inspect
    compare_with : str or list
        Variable(s) to plot for comparison
    window : tuple of str, optional
        (start, end) time window to zoom into
    figsize : tuple of float, optional
        Figure size (width, height) in inches. Default (11, 7) works well in VS Code.
        Use (14, 8) for browser or larger screens.
    """
    global selected_start_time, selected_end_time

    import matplotlib.dates as mdates

    # Close any existing figures to avoid clutter
    plt.close('all')

    # Reset selected times for new inspection
    selected_start_time = None
    selected_end_time = None

    # Find and load the data file
    file_pattern = f"conventional_raw_{deployment_name}_{pips_name}.nc"
    filepath = Path(PIPS_dir) / file_pattern

    if not filepath.exists():
        print(f"❌ File not found: {filepath}")
        return None

    print(f"Loading {filepath}")
    ds = xr.open_dataset(filepath)

    # Check if variable exists
    if variable not in ds:
        print(f"❌ Variable '{variable}' not found in dataset")
        print(f"Available variables: {list(ds.data_vars)}")
        return None

    # Convert xarray time to pandas datetime and then to matplotlib date numbers
    time_pd = pd.to_datetime(ds.time.values)
    time_mpl = mdates.date2num(time_pd)

    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)
    fig.suptitle(f'{pips_name} - {deployment_name}: {variable} Inspection',
                 fontsize=14, fontweight='bold')

    # Plot primary variable using matplotlib date numbers
    ax1.plot(time_mpl, ds[variable].values, label=variable, lw=0.8)

    # Plot comparison variable(s)
    if not isinstance(compare_with, list):
        compare_with = [compare_with]

    for comp_var in compare_with:
        if comp_var in ds:
            ax1.plot(time_mpl, ds[comp_var].values, label=comp_var, lw=0.8, alpha=0.7)

    # ax1.set_ylabel('Temperature (°C)')
    ax1.set_ylabel(f"{variable} {ds[variable].attrs['units']}")
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f"{variable} {ds[variable].attrs['units']}")

    # Plot difference (if comparing with one variable)
    if len(compare_with) == 1 and compare_with[0] in ds:
        diff = ds[variable].values - ds[compare_with[0]].values
        ax2.plot(time_mpl, diff, label=f'{variable} - {compare_with[0]}', color='gray', lw=0.8)
        ax2.axhline(0, color='black', ls='--', lw=1)
        ax2.axhline(2, color='red', ls=':', lw=1, alpha=0.5, label='±2°C threshold')
        ax2.axhline(-2, color='red', ls=':', lw=1, alpha=0.5)
        ax2.set_ylabel('Difference (°C)')
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)
        # ax2.set_title('Temperature Difference')
        ax2.set_title(f"{variable}-{compare_with[0]} Difference")
    else:
        # Plot just the primary variable in bottom panel too
        ax2.plot(time_mpl, ds[variable].values, label=variable, lw=0.8)
        # ax2.set_ylabel('Temperature (°C)')
        ax2.set_ylabel(f"{variable} {ds[variable].attrs['units']}")
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)

    # Format x-axis as dates with proper locator and formatter
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d\n%H:%M:%S'))
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d\n%H:%M:%S'))
    ax2.xaxis.set_major_locator(mdates.AutoDateLocator())

    # Set x-axis limits to actual data range (fixes 1970 epoch issue)
    if window:
        window_start = mdates.date2num(pd.to_datetime(window[0]))
        window_end = mdates.date2num(pd.to_datetime(window[1]))
        ax1.set_xlim(window_start, window_end)
    else:
        # Explicitly set to data range (with a 1 minute buffer on each side for better visualization)
        buffer = 1 / (24 * 60)  # 1 minute in days
        ax1.set_xlim(time_mpl[0] - buffer, time_mpl[-1] + buffer)

    fig.autofmt_xdate()  # Rotate date labels

    # Add text annotations for hover info and selected times
    hover_text = ax1.text(0.02, 0.98, '', transform=ax1.transAxes,
                         verticalalignment='top', fontsize=10,
                         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    selection_text = ax1.text(0.98, 0.98, '', transform=ax1.transAxes,
                             verticalalignment='top', horizontalalignment='right',
                             fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

    # Vertical lines for selected times
    start_line = ax1.axvline(x=0, color='green', ls='--', lw=2, visible=False, label='Start')
    end_line = ax1.axvline(x=0, color='red', ls='--', lw=2, visible=False, label='End')
    start_line2 = ax2.axvline(x=0, color='green', ls='--', lw=2, visible=False)
    end_line2 = ax2.axvline(x=0, color='red', ls='--', lw=2, visible=False)

    def on_motion(event):
        """Display time under cursor."""
        if event.inaxes in [ax1, ax2] and event.xdata is not None:
            hover_time = mdates.num2date(event.xdata)
            hover_text.set_text(f'Time: {hover_time.strftime("%Y-%m-%d %H:%M:%S")}')
            fig.canvas.draw_idle()

    def on_click(event):
        """Record time on click."""
        global selected_start_time, selected_end_time

        if event.inaxes not in [ax1, ax2] or event.xdata is None:
            return

        click_time = mdates.num2date(event.xdata)
        # Convert to pandas Timestamp for consistency
        click_time = pd.Timestamp(click_time)

        # Right click: clear selection
        if event.button == 3:
            selected_start_time = None
            selected_end_time = None
            start_line.set_visible(False)
            end_line.set_visible(False)
            start_line2.set_visible(False)
            end_line2.set_visible(False)
            selection_text.set_text('Selection cleared\nLeft click to select times')
            print("\n✓ Selection cleared")
            fig.canvas.draw_idle()
            return

        # Left click: set start or end time
        if event.button == 1:
            if selected_start_time is None:
                # First click - set start time
                selected_start_time = click_time
                start_line.set_xdata([event.xdata, event.xdata])
                start_line.set_visible(True)
                start_line2.set_xdata([event.xdata, event.xdata])
                start_line2.set_visible(True)
                selection_text.set_text(f'START: {selected_start_time.strftime("%H:%M:%S")}\n'
                                       f'Click again for END time')
                print(f"\n✓ Start time: {selected_start_time}")
            elif selected_end_time is None:
                # Second click - set end time
                selected_end_time = click_time

                # Ensure start < end
                if selected_end_time < selected_start_time:
                    selected_start_time, selected_end_time = selected_end_time, selected_start_time
                    print("  (Times swapped to ensure start < end)")

                end_line.set_xdata([event.xdata, event.xdata])
                end_line.set_visible(True)
                end_line2.set_xdata([event.xdata, event.xdata])
                end_line2.set_visible(True)
                selection_text.set_text(f'START: {selected_start_time.strftime("%H:%M:%S")}\n'
                                       f'END:   {selected_end_time.strftime("%H:%M:%S")}\n'
                                       f'Right click to clear')

                print(f"✓ End time:   {selected_end_time}")
                print(f"\n{'='*60}")
                print("Use these times with add_manual_qc():")
                print(f"add_manual_qc('{pips_name}', '{deployment_name}', '{variable}',")
                print(f"              '{selected_start_time.isoformat()}',")
                print(f"              '{selected_end_time.isoformat()}',")
                print(f"              reason='your reason here')")
                print(f"{'='*60}")
            else:
                # Already have both - reset and start over
                selected_start_time = click_time
                selected_end_time = None
                start_line.set_xdata([event.xdata, event.xdata])
                start_line.set_visible(True)
                start_line2.set_xdata([event.xdata, event.xdata])
                start_line2.set_visible(True)
                end_line.set_visible(False)
                end_line2.set_visible(False)
                selection_text.set_text(f'START: {selected_start_time.strftime("%H:%M:%S")}\n'
                                       f'Click again for END time')
                print(f"\n✓ New start time: {selected_start_time}")

            fig.canvas.draw_idle()

    # Connect event handlers
    fig.canvas.mpl_connect('motion_notify_event', on_motion)
    fig.canvas.mpl_connect('button_press_event', on_click)

    # Initial instruction text
    selection_text.set_text('Left click to select times\nRight click to clear')

    plt.tight_layout()
    plt.show()

    # View existing QC and trim entries for this dataset
    view_manual_qc(pips_name, deployment_name)
    view_manual_trim(pips_name, deployment_name)

    # Display dataset time range for reference
    print(f"\nDataset time range:")
    print(f"  Start: {time_pd[0]}")
    print(f"  End:   {time_pd[-1]}")
    print(f"  Duration: {len(ds.time)} seconds")

    print(f"\n{'='*60}")
    print("INTERACTIVE CONTROLS:")
    print("  • Hover mouse over plot to see exact time")
    print("  • Left click ONCE to set START time (green line)")
    print("  • Left click AGAIN to set END time (red line)")
    print("  • Right click to clear selection and start over")
    print("  • Use selected times with add_manual_qc() command printed above")
    print(f"{'='*60}")

    return ds


print("Inspection function loaded: inspect_variable(pips, iop, var, compare)")
print("  • Includes interactive time selection with mouse clicks")
print("  • Hover to see time, click to select start/end times")

In [ ]:
# Sequential dataset iterator

# Build a flat list of (pips_name, deployment_name) pairs
dataset_list = list(zip(PIPS_names, deployment_names))
current_dataset_index = 0

# Get unique IOPs for progress reporting
unique_iops = list(dict.fromkeys(deployment_names))  # Preserves order

def next_dataset(variable='slowtemp', compare_with='fasttemp', figsize=(11, 7)):
    """Load next dataset for inspection."""
    global current_dataset_index

    # Check if we've finished all datasets
    if current_dataset_index >= len(dataset_list):
        print("\n" + "="*60)
        print("✓ All datasets inspected!")
        print("="*60)
        print(f"\nTotal QC entries: {sum(len(v) for v in manual_qc_dict.values())}")
        print("\nDon't forget to run: save_manual_qc()")
        return None

    pips_name, deployment_name = dataset_list[current_dataset_index]

    # Calculate progress
    current_iop_name = deployment_name
    iop_number = unique_iops.index(current_iop_name) + 1
    total_iops = len(unique_iops)

    # Count which PIPS we're on for this IOP
    pips_in_this_iop = [p for p, d in dataset_list if d == current_iop_name]
    pips_number = pips_in_this_iop.index(pips_name) + 1
    total_pips_in_iop = len(pips_in_this_iop)

    print(f"\n{'='*60}")
    print(f"Inspecting: {pips_name} - {deployment_name}")
    print(f"Progress: IOP {iop_number}/{total_iops}, "
          f"PIPS {pips_number}/{total_pips_in_iop} in this IOP, "
          f"Overall {current_dataset_index + 1}/{len(dataset_list)}")
    print(f"{'='*60}")

    current_dataset_index += 1

    return inspect_variable(pips_name, deployment_name, variable=variable,
                           compare_with=compare_with, figsize=figsize)


def reset_iterator():
    """Reset the dataset iterator to start from beginning."""
    global current_dataset_index
    current_dataset_index = 0
    print("Iterator reset to beginning")


print("Iterator functions loaded:")
print("  next_dataset() - Load next PIPS/IOP combination")
print("  reset_iterator() - Start over from beginning")

## Start Inspection Workflow

Run the cell below to begin inspecting datasets. For each plot:

1. **Examine the data** - Use matplotlib zoom/pan tools to identify problematic regions
2. **Note time ranges** - Identify exact start and end times that need QC
3. **Record decisions** - Use `add_manual_qc()` in next cell
4. **Save periodically** - Run `save_manual_qc()` every few datasets
5. **Continue** - Run `next_dataset()` again

### Example QC entry:
```python
add_manual_qc('PIPS1A', 'IOP1_2025', 'slowtemp', 
              '2025-01-15T10:30:00', '2025-01-15T11:45:00',
              reason='Sensor malfunction - erratic readings')
```

In [ ]:
# Start inspection - load first dataset
ds = next_dataset(figsize=(9, 6))

In [ ]:
# ds = inspect_variable('PIPS1B', 'IOP21_062225', 'dewpoint')

In [ ]:
# Use this cell to add QC decisions after inspecting the plot above
# Example:
# add_manual_qc('PIPS1A', 'IOP1_2025', 'slowtemp',
#               '2025-01-15T10:30:00', '2025-01-15T11:45:00',
#               reason='Sensor malfunction during storm')

# After adding QC entries, save your work:
# save_manual_qc()

# Then continue to next dataset:
# ds = next_dataset()

add_manual_qc(ds.probe_name, ds.deployment_name, 'slowtemp',
              selected_start_time, selected_end_time,
              reason='Sensor malfunction')
save_manual_qc()

In [ ]:
print(selected_start_time, selected_end_time)

In [ ]:
# View all QC decisions at any time
view_manual_qc()

In [ ]:
# Save QC decisions (run this periodically!)
save_manual_qc()

## Advanced Usage

### Inspect specific dataset:
```python
ds = inspect_variable('PIPS3A', 'IOP12_2025', 'slowtemp', 'fasttemp')
```

### Zoom to specific time window:
```python
ds = inspect_variable('PIPS1A', 'IOP1_2025', 'slowtemp', 'fasttemp',
                      window=('2025-01-15 10:00', '2025-01-15 12:00'))
```

### Delete last QC entry (if you made a mistake):
```python
delete_last_qc('PIPS1A', 'IOP1_2025')
```

### After completing all inspections:
1. Run `save_manual_qc()` one final time
2. Apply QC decisions using the command-line script:
   ```bash
   python analysis_scripts/apply_manual_qc.py \
       configs/your_config.py \
       configs/manual_qc_decisions.json
   ```

## Manual Dataset Trimming

Use these functions to trim entire datasets to new time ranges (e.g., remove deployment/retrieval periods).

**Note**: Trimming affects ALL variables, not just specific ones like manual QC.

### Example Usage:

```python
# View current dataset time range
ds = next_dataset('PIPS3A', 'IOP1_2025')
# Shows: Start: 2025-01-15 12:00:00, End: 2025-01-15 18:30:45

# Add manual trim - remove first 5 minutes and last 2 minutes
add_manual_trim('PIPS3A', 'IOP1_2025', 
                new_start_time='2025-01-15 12:05:00',
                new_end_time='2025-01-15 18:28:45',
                reason='Remove deployment and retrieval periods')

# View all trim decisions
view_manual_trim()

# Save decisions to JSON
save_manual_qc()

# If needed, delete trim decision
# delete_manual_trim('PIPS3A', 'IOP1_2025')
```

### Workflow:
1. Use `next_dataset()` or `inspect_variable()` to view data and note time issues
2. Record bad time periods with `add_manual_trim()`
3. Continue inspecting other datasets
4. Save all decisions with `save_manual_qc()` (saves both QC and trim decisions)
5. Apply with batch script: `python apply_manual_qc.py config.py decisions.json`

The batch script will:
- Trim datasets to specified time ranges
- Update global attributes (`starting_time`, `ending_time`)
- Update time coordinate encoding
- Apply to both conventional and parsivel datasets

In [ ]:
reset_iterator()

In [ ]:
# Example: Add manual trimming
# Uncomment and modify for your dataset:

# ds = next_dataset('PIPS3A', 'IOP1_2025')
# print(f"Current time range: {pd.to_datetime(ds.time.values[0])} to {pd.to_datetime(ds.time.values[-1])}")
ds = next_dataset(variable='compass_dir', compare_with=None, figsize=(9, 6))

# # Trim first 30 seconds and last 60 seconds
# start_time = pd.to_datetime(ds.time.values[0]) + pd.Timedelta(seconds=30)
# end_time = pd.to_datetime(ds.time.values[-1]) - pd.Timedelta(seconds=60)

# add_manual_trim('PIPS3A', 'IOP1_2025',
#                 new_start_time=start_time,
#                 new_end_time=end_time,
#                 reason='Remove deployment periods')

# # View and save
# view_manual_trim()
# save_manual_qc()

In [ ]:
add_manual_trim(ds.probe_name, ds.deployment_name,
                new_start_time=selected_start_time,
                new_end_time=selected_end_time,
                reason='Removed early adjustment period')
# add_manual_trim(ds.probe_name, ds.deployment_name,
#                 new_start_time=None,
#                 new_end_time=selected_end_time,
#                 reason='Removed early adjustment period')

In [ ]:
# Save decisions to JSON
save_manual_qc()

In [ ]:
ds = inspect_variable('PIPS3A', 'IOP3_052325', 'GPS_spd',
                      compare_with=None, figsize=(9, 6))

In [ ]:
ds